In [1]:
import pandas as pd
import datetime
import sqlite3
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import sqlalchemy as sqa
import matplotlib.pyplot as plt
import python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

In [6]:
con1 = pymysql.connect(host="192.168.5.124",user="eigyou_kikaku",password="As6hV2K!k",database="eigyou_kikaku",port=3306,charset='cp932')
shokikosho_query = """
select
	A.kohosha_id as '候補者id',
	case 
	when A.ap_source = 10 then 'パートナー紹介'
	END as '候補者APソース',
	B.sei_plus as 'アポ獲得者',
	A.kosho_setteibi as '初期交渉設定日',
	A.kosho_seq as '累計設定回数'
from
	_live_rhs__shokikoshos A
left join
	_live_company__syain B
on
	A.ap_kakutoku = B.user_id
where
  kosho_setteibi >= date '2022/10/01'
and
  kosho_seq = 1
and
	ap_source = 10
order by
	kosho_setteibi asc
;
"""
rzshokikosho = pd.read_sql(shokikosho_query,con1)
con1.close()

rzshokikosho["初期交渉設定日"] = rzshokikosho["初期交渉設定日"].astype(str) 
rzshokikosho["累計設定回数"] = rzshokikosho["累計設定回数"].astype(str) 

rzshokikosho.replace([np.inf, -np.inf], np.nan, inplace=True)
rzshokikosho.fillna('', inplace=True)
rzshokikosho = rzshokikosho.values.tolist()

type(rzshokikosho)

C:\Users\suehara\Anaconda3\lib\site-packages\pandas\io\sql.py:762: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


list

In [7]:
# 初期交渉DBシートにコピペ
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1SXaqbu_eoPHVSrqeZuXU84eAgFO5H1jpjAzIhuexwKk'

In [8]:
service = ps.get_auth(SCOPES,json_path)
Sheet_NAME = 'pythonてすと!'
Sheet_row = "A2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzshokikosho,service)

In [9]:
con1 = pymysql.connect(host="192.168.5.124",user="eigyou_kikaku",password="As6hV2K!k",database="eigyou_kikaku",port=3306,charset='cp932')
honkosho_query = """
select
	A.id as '本交渉id',
	B.sei_plus as '候補者担当（2）',
	D.name,
	E.sei_plus as '企業担当',
	A.kosho_setteibi as '本交渉設定日',
	A.kosho_seq as '設定回数'
from
	_live_rhs__honkoshos A
left join
	_live_company__syain B
on
	A.kohosha_tanto = B.user_id
left join 
	_live_rhs__ankens	C
on
	A.anken_id = C.id
left join 
	_live_rhs__kigyos D
on
 C.kigyo_id = D.id
left join
	_live_company__syain E
on
	A.mendan_tanto = E.user_id
where
	A.kosho_seq = 1
and 
	A.kosho_setteibi >= date '2022/10/01'
order by 
	A.kosho_setteibi asc
;
"""
rzhonkosho = pd.read_sql(honkosho_query,con1)
con1.close()

rzhonkosho["本交渉設定日"] = rzhonkosho["本交渉設定日"].astype(str) 
rzhonkosho["設定回数"] = rzhonkosho["設定回数"].astype(str) 

rzhonkosho.replace([np.inf, -np.inf], np.nan, inplace=True)
rzhonkosho.fillna('', inplace=True)
rzhonkosho = rzhonkosho.values.tolist()

type(rzhonkosho)


C:\Users\suehara\Anaconda3\lib\site-packages\pandas\io\sql.py:762: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


list

In [10]:
# 初期交渉DBシートにコピペ
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1SXaqbu_eoPHVSrqeZuXU84eAgFO5H1jpjAzIhuexwKk'

In [11]:
service = ps.get_auth(SCOPES,json_path)
Sheet_NAME = 'pythonてすと!'
Sheet_row = "G2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzhonkosho,service)